# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Load Data

In [13]:
# List all files
files <- list.files(path = "09_WGCNA_GO", pattern = "_enrichr_results\\.xlsx$", full.names = TRUE)

# Extract module names from filenames and filter
module_names <- gsub("09_WGCNA_GO/(.*)_enrichr_results\\.xlsx", "\\1", files)

# View the module names
print(module_names)

 [1] "antiquewhite4"   "bisque4"         "black"           "blue"           
 [5] "brown"           "brown4"          "coral1"          "coral2"         
 [9] "cyan"            "darkgreen"       "darkgrey"        "darkmagenta"    
[13] "darkolivegreen"  "darkorange"      "darkorange2"     "darkred"        
[17] "darkseagreen4"   "darkslateblue"   "darkturquoise"   "floralwhite"    
[21] "green"           "greenyellow"     "grey"            "grey60"         
[25] "honeydew1"       "ivory"           "lavenderblush3"  "lightcyan"      
[29] "lightcyan1"      "lightgreen"      "lightpink4"      "lightsteelblue1"
[33] "lightyellow"     "magenta"         "maroon"          "mediumorchid"   
[37] "mediumpurple3"   "midnightblue"    "navajowhite2"    "orange"         
[41] "orangered3"      "orangered4"      "paleturquoise"   "palevioletred3" 
[45] "pink"            "plum"            "plum1"           "plum2"          
[49] "purple"          "red"             "royalblue"       "saddlebrown"    

# Combined Dot Plot

In [14]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Reactome_2022") #, "RNAseq_DiseaseGene_DrugSigs_GEO")

In [15]:
# Create abbreviation mapping
database_abbreviations <- c(
  "GO_Biological_Process_2025" = "GO BP",
  "GO_Cellular_Component_2025" = "GO CC", 
  "GO_Molecular_Function_2025" = "GO MF",
  "KEGG_2021_Human" = "KEGG",
  "Reactome_2022" = "REACTOME"
  #"RNAseq_DiseaseGene_DrugSigs_GEO" = "GEO RNAseq"
)

In [16]:
# Initialize an empty dataframe for all GO data
all_data <- data.frame()

In [17]:
# Iterate over each module and read in the corresponding Excel file
for (file in files) {
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        filter(Adjusted.P.value <= 0.05) %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = file, Database = database) %>%
        arrange(Adjusted.P.value)
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Spliceosomal Complex Assembly (GO:0000245),0.024890663,20.10625,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,GO_Biological_Process_2025
2,Regulation of Toll-Like Receptor Signaling Pathway (GO:0034121),0.024890663,39.92045,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,GO_Biological_Process_2025
3,mRNA Processing (GO:0006397),0.026917656,8.26951,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,GO_Biological_Process_2025
4,Processing Of Capped Intron-Containing Pre-mRNA R-HSA-72203,0.004804918,9.16609,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
5,mRNA Splicing - Minor Pathway R-HSA-72165,0.004804918,26.38010,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
6,mRNA Splicing - Major Pathway R-HSA-72163,0.004964849,10.41969,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022


In [18]:
# Format comparison name
all_data$Module <- gsub("09_WGCNA_GO/", "", all_data$Module)
all_data$Module <- gsub("_enrichr_results.xlsx", "", all_data$Module)

# View
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Spliceosomal Complex Assembly (GO:0000245),0.024890663,20.10625,darkolivegreen,GO_Biological_Process_2025
2,Regulation of Toll-Like Receptor Signaling Pathway (GO:0034121),0.024890663,39.92045,darkolivegreen,GO_Biological_Process_2025
3,mRNA Processing (GO:0006397),0.026917656,8.26951,darkolivegreen,GO_Biological_Process_2025
4,Processing Of Capped Intron-Containing Pre-mRNA R-HSA-72203,0.004804918,9.16609,darkolivegreen,Reactome_2022
5,mRNA Splicing - Minor Pathway R-HSA-72165,0.004804918,26.38010,darkolivegreen,Reactome_2022
6,mRNA Splicing - Major Pathway R-HSA-72163,0.004964849,10.41969,darkolivegreen,Reactome_2022


In [20]:
# Generate dot plot
if (nrow(all_data) > 0) {
  
  # Create database abbreviations and assign colors
  all_data <- all_data %>%
    mutate(
      DB_Abbrev = ifelse(Database %in% names(database_abbreviations), 
                         database_abbreviations[Database], Database),
      Module = factor(Module),
      # Create simple term label with database
      Simple_Term = paste(DB_Abbrev, Term, sep = ": ")
    )
  
  # Create a color mapping for databases
  unique_dbs <- unique(all_data$DB_Abbrev)
  db_colors <- setNames(RColorBrewer::brewer.pal(max(3, length(unique_dbs)), "Set3")[1:length(unique_dbs)], 
                       unique_dbs)
  
  # Sort terms by database first, then by term name
  all_data <- all_data %>%
    arrange(DB_Abbrev, Term)
  
  # Ensure terms are ordered properly
  term_order <- unique(all_data$Simple_Term)
  all_data$Simple_Term <- factor(all_data$Simple_Term, levels = term_order)
  
  dot_plot <- ggplot(all_data, aes(x = Module, y = Simple_Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
    geom_point(shape = 21, alpha = 0.8) +
    scale_fill_viridis(name = 'Adjusted P-value', direction = -1) +
    scale_size_continuous(name = 'Odds Ratio', range = c(1, 8)) +
    xlab('') + ylab('') +
    labs(title = 'Significant GO Terms for WGCNA Modules') +
    theme_minimal() +
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_line(color = "black"),
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
      axis.text.y = element_text(size = 7),
      legend.position = "right"
    )
  
  # Save the plot
  ggsave(filename = "09_WGCNA_GO/combined_go_dot_plot_adjpval.pdf", 
         plot = dot_plot, height = 10, width = 10)
}